In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [2]:

def list_other_pollutants(id_mma, monitoramento_qar):

    filtro = monitoramento_qar[monitoramento_qar['ID_MMA'] == id_mma]

    filtro = filtro[~filtro['POLUENTE'].isin(['MP10', 'MP25', 'NO2'])]
    
    poluentes_restantes = ', '.join(filtro['POLUENTE'].unique())
    
    return poluentes_restantes

df_oms = pd.DataFrame(columns=['country_name','city_orig','city_code_orig','station_name','station_id_orig','type_of_station_orig',
                          'type_of_station_detailed','latitude','longitude','adress','year',
                           'pm10_concentration','pm10_tempcov','pm10_tempcov_days','equipment_pm10','pop_cov_pm10',
                           'pm25_concentration','pm25_tempcov','pm25_tempcov_days','equipment_pm25','pop_cov_pm25',
                           'no2_concentration','no2_tempcov','no2_tempcov_days','equipment_no2','pop_cov_no2',
                           'source_web_link','other_pollutants','population','population_source','pop_year','comments'])

pastas = ['MP10','MP25','NO2']

pop_buffer = pd.read_csv(os.getcwd()+'/data/outputs/populacao_varbuf.csv')

pop_mun = pd.read_csv(os.getcwd()+'/data/dicionarios/br_ibge_populacao_municipio.csv')
pop_mun = pop_mun[pop_mun['ano']==2022]

type_station = pd.read_csv(os.getcwd()+'/data/outputs/uso_solo_varbuf.csv')
type_station['rural'] = type_station['Herbácea_perc']+type_station['Agropecuária_perc']
type_station['urban'] = type_station['Urbanizada_perc']
type_station['perc_urban'] = type_station['urban']/(type_station['urban']+type_station['rural'])

monitoramento_qar = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')
monitoramento_pols = monitoramento_qar[monitoramento_qar['POLUENTE'].isin(['MP10','MP25','NO2'])]

list_id_mma = monitoramento_pols['ID_MMA'].unique()

for id_mma in list_id_mma:

    dados_estacao = monitoramento_pols.loc[monitoramento_pols['ID_MMA'] == id_mma, 
            ['CIDADE','CD_MUN', 'ID_OEMA', 'ID_MMA', 'LATITUDE', 'LONGITUDE']].reset_index()

    try:
        perc_solo = type_station[type_station['ID_MMA']==id_mma]['perc_urban'].reset_index()['perc_urban'][0]

        if perc_solo >= 0.7:
            tipo_solo = 'urban'
        elif perc_solo <= 0.4:
            tipo_solo = 'rural'
        else:
            tipo_solo = 'sub-urban'
        
    except:
        tipo_solo = 'Não há dado de localização da estação'

    try:
        populacao = int(pop_mun[pop_mun['id_municipio']==int(dados_estacao['CD_MUN'][0])]['populacao'].reset_index()['populacao'][0])
    except:
        populacao = 'Não há dados de população'

    list_pollutants = list_other_pollutants(id_mma, monitoramento_qar)
    
    linha_geral = {'country_name':['Brasil'],
                 'city_orig':[dados_estacao['CIDADE'][0]],
                 'city_code_orig':[dados_estacao['CD_MUN'][0]],
                 'station_name':[dados_estacao['ID_OEMA'][0]],
                 'station_id_orig':[id_mma],
                 'type_of_station_orig':[tipo_solo],
                 'type_of_station_detailed':[''],
                 'latitude':[dados_estacao['LATITUDE'][0]],
                 'longitude':[dados_estacao['LONGITUDE'][0]],
                 'year':[''],
                 'pm10_concentration':[''],
                 'pm10_tempcov':[''],
                 'equipment_pm10':[''],
                 'pop_cov_pm10':[''],     
                 'pm25_concentration':[''],
                 'pm25_tempcov':[''],
                 'equipment_pm25':[''],
                 'pop_cov_pm25':[''],      
                 'no2_concentration':[''],
                 'no2_tempcov':[''],
                 'equipment_no2':[''],
                 'pop_cov_no2':[''],
                 'source_web_link':['https://hoinaski.prof.ufsc.br/files/'],
                 'other_pollutants':[list_pollutants],
                 'population':[populacao],
                 'population_source':['IBGE'],
                 'pop_year':[2025],
                 'comments':['']}

    dict_rep_espacial={
                        'MP10': [np.nan],
                        'MP25': [np.nan],
                        'NO2': [np.nan]
                    }

    for ano in range(2010,2025):

        valor = 0

        linha_geral_ano = linha_geral

        linha_geral_ano['year'] = [ano]
            
        for pol in pastas:
    
            if pol == 'MP10':
                variavel = 'pm10'
            elif pol == 'MP25':
                variavel = 'pm25'
            elif pol == 'NO2':
                variavel = 'no2'
    
            monitoramento_pol = monitoramento_pols[monitoramento_pols['POLUENTE']==pol]
            monitoramento_estacao = monitoramento_pol[monitoramento_pol['ID_MMA']==id_mma].reset_index() 

            if len(monitoramento_estacao) > 0:
                
                id_mma_completo = monitoramento_estacao['ID_MMA_COMPLETO'][0]

                try:
                    pop_cov = pop_buffer[pop_buffer['ID_MMA_COMPLETO']==id_mma_completo]['POP_BUFFER'].reset_index()['POP_BUFFER'][0]
                except:
                    pop_cov = 'Sem dados de cobertura da estação'
                
                try:
                    equipment = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['MODELO'].reset_index()['MODELO'][0]
                except:
                    equipment = 'Sem dados de equipamento'

                dict_rep_espacial[pol] = ['A representatividade espacial de ' + pol + ' é ' + monitoramento_estacao['REP_ESPACIAL'][0]]
                
                rep_espacial = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['REP_ESPACIAL'].reset_index()['REP_ESPACIAL'][0]
                
                if (id_mma_completo+'.csv') in os.listdir(os.getcwd()+'/data/MQAr_averages/anual/'+pol):

                    df = pd.read_csv(os.getcwd()+'/data/MQAr_averages/anual/'+pol+'/'+id_mma_completo+'.csv')

                    df = df[df['ANO']==ano].reset_index()

                    print('Entrou no if: '+id_mma_completo)

                    if len(df)>0:
                        
                        print(ano)

                        valor = valor + 1
                        
                        tempcov = df['PRCNT_DIAS_ANO_REP_TEMPORAL'][0]
                        concentration = df['VALOR'][0]

                        linha_geral_ano[variavel+'_concentration'] = [concentration]
                        linha_geral_ano[variavel+'_tempcov'] = [tempcov]
                        linha_geral_ano['equipment_'+variavel] = [equipment]
                        if pop_cov == 'Sem dados de cobertura da estação':                        
                            linha_geral_ano['pop_cov_'+variavel] = ['Sem dados de cobertura da estação']
                        else:
                            linha_geral_ano['pop_cov_'+variavel] = [int(pop_cov)]

        if valor > 0:

            rep_espacial = [v for v in dict_rep_espacial.values() if pd.notna(v)]

            rep_espacial = ', '.join([item[0] for item in rep_espacial])

            linha_geral_ano['type_of_station_detailed'] = rep_espacial
            
            df_linha = pd.DataFrame(linha_geral_ano)
            
            df_oms = pd.concat([df_oms, df_linha], ignore_index=True)
            

Entrou no if: BA0010ND001
2010
Entrou no if: BA0010ND004
2010
Entrou no if: BA0010ND001
2011
Entrou no if: BA0010ND004
2011
Entrou no if: BA0010ND001
2012
Entrou no if: BA0010ND004
2012
Entrou no if: BA0010ND001
2013
Entrou no if: BA0010ND004
2013
Entrou no if: BA0010ND001
2014
Entrou no if: BA0010ND004
2014
Entrou no if: BA0010ND001
2015
Entrou no if: BA0010ND004
2015
Entrou no if: BA0010ND001
2016
Entrou no if: BA0010ND004
2016
Entrou no if: BA0010ND001
2017
Entrou no if: BA0010ND004
2017
Entrou no if: BA0010ND001
2018
Entrou no if: BA0010ND004
2018
Entrou no if: BA0010ND001
2019
Entrou no if: BA0010ND004
2019
Entrou no if: BA0010ND001
2020
Entrou no if: BA0010ND004
2020
Entrou no if: BA0010ND001
2021
Entrou no if: BA0010ND004
2021
Entrou no if: BA0010ND001
2022
Entrou no if: BA0010ND004
2022


/tmp/ipykernel_13427/661489358.py:168: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_oms = pd.concat([df_oms, df_linha], ignore_index=True)


Entrou no if: BA0010ND001
2023
Entrou no if: BA0010ND004
2023
Entrou no if: BA0010ND001
2024
Entrou no if: BA0010ND004
2024
Entrou no if: BA0013ND001
2010
Entrou no if: BA0013ND004
2010
Entrou no if: BA0013ND001
2011
Entrou no if: BA0013ND004
2011
Entrou no if: BA0013ND001
2012
Entrou no if: BA0013ND004
2012
Entrou no if: BA0013ND001
2013
Entrou no if: BA0013ND004
2013
Entrou no if: BA0013ND001
2014
Entrou no if: BA0013ND004
2014
Entrou no if: BA0013ND001
2015
Entrou no if: BA0013ND004
2015
Entrou no if: BA0013ND001
2016
Entrou no if: BA0013ND004
2016
Entrou no if: BA0013ND001
2017
Entrou no if: BA0013ND004
2017
Entrou no if: BA0013ND001
2018
Entrou no if: BA0013ND004
2018
Entrou no if: BA0013ND001
2019
Entrou no if: BA0013ND004
2019
Entrou no if: BA0013ND001
2020
Entrou no if: BA0013ND004
2020
Entrou no if: BA0013ND001
2021
Entrou no if: BA0013ND004
2021
Entrou no if: BA0013ND001
2022
Entrou no if: BA0013ND004
2022
Entrou no if: BA0013ND001
2023
Entrou no if: BA0013ND004
2023
Entrou n

KeyboardInterrupt: 

In [5]:
monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['MODELO']

0    Nao declarado
Name: MODELO, dtype: object

In [46]:
type_station[type_station['CIDADE']=='Porto Alegre']

,Unnamed: 0,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,...,Floresta_perc,Herbácea_perc,Agropecuária_perc,Não Vegetada_perc,Urbanizada_perc,Mineração_perc,GRUPO_PRED_VAR,rural,urban,perc_urban
182,182,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA007,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
183,183,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA007,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
408,210,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA004,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
416,218,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA004,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
660,221,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA005,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
661,222,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA005,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1486,794,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA001,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1487,795,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA002,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1489,797,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA002,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1731,196,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA003,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127


In [32]:
dados_estacao['CD_MUN'][0]

'1505536'

In [50]:
df_oms.to_csv(os.getcwd()+'/data/dicionarios/envio_oms.csv', index=False)

,country_name,city_orig,city_code_orig,station_name,station_id_orig,type_of_station_orig,type_of_station_detailed,latitude,longitude,adress,...,no2_tempcov,no2_tempcov_days,equipment_no2,pop_cov_no2,source_web_link,other_pollutants,population,population_source,pop_year,comments
0,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
1,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
2,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
3,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
4,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,10.382514,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
161,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,72.054795,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
162,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,77.534247,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
163,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,98.082192,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,


In [13]:
'''df_monitoramento = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')

df_codigo = pd.read_csv(os.getcwd()+'/data/dicionarios/CODIGO_POLUENTES.csv')

df_monitoramento = df_monitoramento.drop_duplicates(subset=['ID_MMA', 'UF', 'POLUENTE'], keep='first')

df_monitoramento = df_monitoramento.merge(
    df_codigo[['COD_POLUENTE', 'POLUENTE']],
    on=['COD_POLUENTE', 'POLUENTE'],
    how='inner'
)

df_monitoramento'''

"df_monitoramento = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')\n\ndf_codigo = pd.read_csv(os.getcwd()+'/data/dicionarios/CODIGO_POLUENTES.csv')\n\ndf_monitoramento = df_monitoramento.drop_duplicates(subset=['ID_MMA', 'UF', 'POLUENTE'], keep='first')\n\ndf_monitoramento = df_monitoramento.merge(\n    df_codigo[['COD_POLUENTE', 'POLUENTE']],\n    on=['COD_POLUENTE', 'POLUENTE'],\n    how='inner'\n)\n\ndf_monitoramento"

In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [2]:

def list_other_pollutants(id_mma, monitoramento_qar):

    filtro = monitoramento_qar[monitoramento_qar['ID_MMA'] == id_mma]

    filtro = filtro[~filtro['POLUENTE'].isin(['MP10', 'MP25', 'NO2'])]
    
    poluentes_restantes = ', '.join(filtro['POLUENTE'].unique())
    
    return poluentes_restantes

df_oms = pd.DataFrame(columns=['country_name','city_orig','city_code_orig','station_name','station_id_orig','type_of_station_orig',
                          'type_of_station_detailed','latitude','longitude','adress','year',
                           'pm10_concentration','pm10_tempcov','pm10_tempcov_days','equipment_pm10','pop_cov_pm10',
                           'pm25_concentration','pm25_tempcov','pm25_tempcov_days','equipment_pm25','pop_cov_pm25',
                           'no2_concentration','no2_tempcov','no2_tempcov_days','equipment_no2','pop_cov_no2',
                           'source_web_link','other_pollutants','population','population_source','pop_year','comments'])

pastas = ['MP10','MP25','NO2']

pop_buffer = pd.read_csv(os.getcwd()+'/data/outputs/populacao_varbuf.csv')

pop_mun = pd.read_csv(os.getcwd()+'/data/dicionarios/br_ibge_populacao_municipio.csv')
pop_mun = pop_mun[pop_mun['ano']==2022]

type_station = pd.read_csv(os.getcwd()+'/data/outputs/uso_solo_varbuf.csv')
type_station['rural'] = type_station['Herbácea_perc']+type_station['Agropecuária_perc']
type_station['urban'] = type_station['Urbanizada_perc']
type_station['perc_urban'] = type_station['urban']/(type_station['urban']+type_station['rural'])

monitoramento_qar = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')
monitoramento_pols = monitoramento_qar[monitoramento_qar['POLUENTE'].isin(['MP10','MP25','NO2'])]

list_id_mma = monitoramento_pols['ID_MMA'].unique()

for id_mma in list_id_mma:

    dados_estacao = monitoramento_pols.loc[monitoramento_pols['ID_MMA'] == id_mma, 
            ['CIDADE','CD_MUN', 'ID_OEMA', 'ID_MMA', 'LATITUDE', 'LONGITUDE']].reset_index()

    try:
        perc_solo = type_station[type_station['ID_MMA']==id_mma]['perc_urban'].reset_index()['perc_urban'][0]

        if perc_solo >= 0.7:
            tipo_solo = 'urban'
        elif perc_solo <= 0.4:
            tipo_solo = 'rural'
        else:
            tipo_solo = 'sub-urban'
        
    except:
        tipo_solo = 'Não há dado de localização da estação'

    try:
        populacao = int(pop_mun[pop_mun['id_municipio']==int(dados_estacao['CD_MUN'][0])]['populacao'].reset_index()['populacao'][0])
    except:
        populacao = 'Não há dados de população'

    list_pollutants = list_other_pollutants(id_mma, monitoramento_qar)
    
    linha_geral = {'country_name':['Brasil'],
                 'city_orig':[dados_estacao['CIDADE'][0]],
                 'city_code_orig':[dados_estacao['CD_MUN'][0]],
                 'station_name':[dados_estacao['ID_OEMA'][0]],
                 'station_id_orig':[id_mma],
                 'type_of_station_orig':[tipo_solo],
                 'type_of_station_detailed':[''],
                 'latitude':[dados_estacao['LATITUDE'][0]],
                 'longitude':[dados_estacao['LONGITUDE'][0]],
                 'year':[''],
                 'pm10_concentration':[''],
                 'pm10_tempcov':[''],
                 'equipment_pm10':[''],
                 'pop_cov_pm10':[''],     
                 'pm25_concentration':[''],
                 'pm25_tempcov':[''],
                 'equipment_pm25':[''],
                 'pop_cov_pm25':[''],      
                 'no2_concentration':[''],
                 'no2_tempcov':[''],
                 'equipment_no2':[''],
                 'pop_cov_no2':[''],
                 'source_web_link':['https://hoinaski.prof.ufsc.br/files/'],
                 'other_pollutants':[list_pollutants],
                 'population':[populacao],
                 'population_source':['IBGE'],
                 'pop_year':[2025],
                 'comments':['']}

    dict_rep_espacial={
                        'MP10': [np.nan],
                        'MP25': [np.nan],
                        'NO2': [np.nan]
                    }

    for ano in range(2010,2025):

        valor = 0

        linha_geral_ano = linha_geral

        linha_geral_ano['year'] = [ano]
            
        for pol in pastas:
    
            if pol == 'MP10':
                variavel = 'pm10'
            elif pol == 'MP25':
                variavel = 'pm25'
            elif pol == 'NO2':
                variavel = 'no2'
    
            monitoramento_pol = monitoramento_pols[monitoramento_pols['POLUENTE']==pol]
            monitoramento_estacao = monitoramento_pol[monitoramento_pol['ID_MMA']==id_mma].reset_index() 

            if len(monitoramento_estacao) > 0:
                
                id_mma_completo = monitoramento_estacao['ID_MMA_COMPLETO'][0]

                try:
                    pop_cov = pop_buffer[pop_buffer['ID_MMA_COMPLETO']==id_mma_completo]['POP_BUFFER'].reset_index()['POP_BUFFER'][0]
                except:
                    pop_cov = 'Sem dados de cobertura da estação'
                
                try:
                    equipment = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['MODELO'].reset_index()['MODELO'][0]
                except:
                    equipment = 'Sem dados de equipamento'

                dict_rep_espacial[pol] = ['A representatividade espacial de ' + pol + ' é ' + monitoramento_estacao['REP_ESPACIAL'][0]]
                
                rep_espacial = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['REP_ESPACIAL'].reset_index()['REP_ESPACIAL'][0]
                
                if (id_mma_completo+'.csv') in os.listdir(os.getcwd()+'/data/MQAr_averages/anual/'+pol):

                    df = pd.read_csv(os.getcwd()+'/data/MQAr_averages/anual/'+pol+'/'+id_mma_completo+'.csv')

                    df = df[df['ANO']==ano].reset_index()

                    print('Entrou no if: '+id_mma_completo)

                    if len(df)>0:
                        
                        print(ano)

                        valor = valor + 1
                        
                        tempcov = df['PRCNT_DIAS_ANO_REP_TEMPORAL'][0]
                        concentration = df['VALOR'][0]

                        linha_geral_ano[variavel+'_concentration'] = [concentration]
                        linha_geral_ano[variavel+'_tempcov'] = [tempcov]
                        linha_geral_ano['equipment_'+variavel] = [equipment]
                        if pop_cov == 'Sem dados de cobertura da estação':                        
                            linha_geral_ano['pop_cov_'+variavel] = ['Sem dados de cobertura da estação']
                        else:
                            linha_geral_ano['pop_cov_'+variavel] = [int(pop_cov)]

        if valor > 0:

            rep_espacial = [v for v in dict_rep_espacial.values() if pd.notna(v)]

            rep_espacial = ', '.join([item[0] for item in rep_espacial])

            linha_geral_ano['type_of_station_detailed'] = rep_espacial
            
            df_linha = pd.DataFrame(linha_geral_ano)
            
            df_oms = pd.concat([df_oms, df_linha], ignore_index=True)
            

Entrou no if: BA0010ND001
2010
Entrou no if: BA0010ND004
2010
Entrou no if: BA0010ND001
2011
Entrou no if: BA0010ND004
2011
Entrou no if: BA0010ND001
2012
Entrou no if: BA0010ND004
2012
Entrou no if: BA0010ND001
2013
Entrou no if: BA0010ND004
2013
Entrou no if: BA0010ND001
2014
Entrou no if: BA0010ND004
2014
Entrou no if: BA0010ND001
2015
Entrou no if: BA0010ND004
2015
Entrou no if: BA0010ND001
2016
Entrou no if: BA0010ND004
2016
Entrou no if: BA0010ND001
2017
Entrou no if: BA0010ND004
2017
Entrou no if: BA0010ND001
2018
Entrou no if: BA0010ND004
2018
Entrou no if: BA0010ND001
2019
Entrou no if: BA0010ND004
2019
Entrou no if: BA0010ND001
2020
Entrou no if: BA0010ND004
2020
Entrou no if: BA0010ND001
2021
Entrou no if: BA0010ND004
2021
Entrou no if: BA0010ND001
2022
Entrou no if: BA0010ND004
2022


/tmp/ipykernel_13427/661489358.py:168: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_oms = pd.concat([df_oms, df_linha], ignore_index=True)


Entrou no if: BA0010ND001
2023
Entrou no if: BA0010ND004
2023
Entrou no if: BA0010ND001
2024
Entrou no if: BA0010ND004
2024
Entrou no if: BA0013ND001
2010
Entrou no if: BA0013ND004
2010
Entrou no if: BA0013ND001
2011
Entrou no if: BA0013ND004
2011
Entrou no if: BA0013ND001
2012
Entrou no if: BA0013ND004
2012
Entrou no if: BA0013ND001
2013
Entrou no if: BA0013ND004
2013
Entrou no if: BA0013ND001
2014
Entrou no if: BA0013ND004
2014
Entrou no if: BA0013ND001
2015
Entrou no if: BA0013ND004
2015
Entrou no if: BA0013ND001
2016
Entrou no if: BA0013ND004
2016
Entrou no if: BA0013ND001
2017
Entrou no if: BA0013ND004
2017
Entrou no if: BA0013ND001
2018
Entrou no if: BA0013ND004
2018
Entrou no if: BA0013ND001
2019
Entrou no if: BA0013ND004
2019
Entrou no if: BA0013ND001
2020
Entrou no if: BA0013ND004
2020
Entrou no if: BA0013ND001
2021
Entrou no if: BA0013ND004
2021
Entrou no if: BA0013ND001
2022
Entrou no if: BA0013ND004
2022
Entrou no if: BA0013ND001
2023
Entrou no if: BA0013ND004
2023
Entrou n

KeyboardInterrupt: 

In [5]:
monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['MODELO']

0    Nao declarado
Name: MODELO, dtype: object

In [46]:
type_station[type_station['CIDADE']=='Porto Alegre']

,Unnamed: 0,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,...,Floresta_perc,Herbácea_perc,Agropecuária_perc,Não Vegetada_perc,Urbanizada_perc,Mineração_perc,GRUPO_PRED_VAR,rural,urban,perc_urban
182,182,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA007,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
183,183,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA007,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
408,210,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA004,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
416,218,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA004,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
660,221,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA005,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
661,222,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA005,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1486,794,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA001,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1487,795,RN,Porto Alegre,4314900,PORTO ALEGRECETE,RN1001,RN1001RA002,Nao declarado,Nao declarado,Nao declarado,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1489,797,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA002,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127
1731,196,RS,Porto Alegre,4314902,POA CETE,RS0022,RS0022RA003,FEPAM,Publica,RB Ambiental,...,1.3,1.3,0.0,16.7,80.6,0.0,Urbanizada,1.3,80.6,0.984127


In [32]:
dados_estacao['CD_MUN'][0]

'1505536'

In [50]:
df_oms.to_csv(os.getcwd()+'/data/dicionarios/envio_oms.csv', index=False)

,country_name,city_orig,city_code_orig,station_name,station_id_orig,type_of_station_orig,type_of_station_detailed,latitude,longitude,adress,...,no2_tempcov,no2_tempcov_days,equipment_no2,pop_cov_no2,source_web_link,other_pollutants,population,population_source,pop_year,comments
0,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
1,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
2,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
3,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
4,Brasil,Salvador,2927408,BOTELHO,BA0010,Não há dado de localização da estação,,-12.782422,-38.515554,NaN,...,0.000000,NaN,Nao declarado,4459,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,10.382514,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
161,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,72.054795,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
162,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,77.534247,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,
163,Brasil,Candeias,2906501,MALEMBA,BA0011,Não há dado de localização da estação,,-12.679219,-38.544115,NaN,...,98.082192,NaN,Nao declarado,53963,https://hoinaski.prof.ufsc.br/files/,"CO, O3, SO2",Não há dados de população,IBGE,2025,


In [13]:
'''df_monitoramento = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')

df_codigo = pd.read_csv(os.getcwd()+'/data/dicionarios/CODIGO_POLUENTES.csv')

df_monitoramento = df_monitoramento.drop_duplicates(subset=['ID_MMA', 'UF', 'POLUENTE'], keep='first')

df_monitoramento = df_monitoramento.merge(
    df_codigo[['COD_POLUENTE', 'POLUENTE']],
    on=['COD_POLUENTE', 'POLUENTE'],
    how='inner'
)

df_monitoramento'''

"df_monitoramento = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')\n\ndf_codigo = pd.read_csv(os.getcwd()+'/data/dicionarios/CODIGO_POLUENTES.csv')\n\ndf_monitoramento = df_monitoramento.drop_duplicates(subset=['ID_MMA', 'UF', 'POLUENTE'], keep='first')\n\ndf_monitoramento = df_monitoramento.merge(\n    df_codigo[['COD_POLUENTE', 'POLUENTE']],\n    on=['COD_POLUENTE', 'POLUENTE'],\n    how='inner'\n)\n\ndf_monitoramento"